# Notebook 08 — Hybrid RAG Chatbot

Combines **SQL retrieval** (structured queries) + **vector search** (semantic context)
+ **Gemini LLM** to produce grounded, natural-language answers about Kerala IDSP data.

**Architecture:**
```
User Question
     |
     v
Intent Classifier --> SQL Query Router --> Structured Results
     |                                          |
     v                                          v
Vector Search --> Semantic Context         Combined Context
                                                |
                                                v
                                          Gemini LLM
                                                |
                                                v
                                       Grounded Answer
```

## 1. Setup

In [ ]:
import os
import sys
from dotenv import load_dotenv
from google import genai

# Add project root to path so we can import our modules
PROJECT_ROOT = os.path.abspath("..")
sys.path.insert(0, PROJECT_ROOT)

from query_router import QueryRouter, classify_intent, extract_district, extract_disease
from vector_search import VectorSearch

load_dotenv(os.path.join(PROJECT_ROOT, ".env"))

DB_PATH = os.path.join("data", "idsp_kerala.db")
CHROMA_PATH = os.path.join(PROJECT_ROOT, "data", "chroma_db")

# Initialize components
router = QueryRouter(DB_PATH)
vector = VectorSearch(CHROMA_PATH, env_path=os.path.join(PROJECT_ROOT, ".env"))
gemini_client = genai.Client(api_key=os.getenv("GEMINI_API_KEY"))

LLM_MODEL = "gemini-3.6-flash"

print("All components initialized.")
print(f"  SQL DB: {DB_PATH}")
print(f"  ChromaDB: {CHROMA_PATH}")
print(f"  LLM: {LLM_MODEL}")

## 2. System Prompt

In [13]:
SYSTEM_PROMPT = """You are an IDSP Kerala Disease Surveillance Assistant. You answer questions
about disease outbreaks, case counts, deaths, and localities in Kerala using
official IDSP (Integrated Disease Surveillance Programme) daily reports.

RULES:
1. Only use the data provided in the context below. Never invent numbers or facts.
2. Always mention the report date so the user knows how current the data is.
3. If the data says 0 cases, say so — do not skip diseases with zero values.
4. Reported cases are preliminary and may change after lab tests and death audits.
   Include this caveat when discussing deaths or confirmed cases.
5. If the context doesn't contain enough information to answer, say so clearly.
6. Keep answers concise but complete. Use bullet points for lists.
7. When mentioning localities, note they come from the IDSP locality section and
   represent where cases were reported, not necessarily the only affected areas.
"""

print("System prompt defined.")

System prompt defined.


## 3. Hybrid Retrieval + LLM Answer

In [14]:
def build_context(question: str) -> str:
    """Combine SQL results and vector search into a single context string."""
    parts = []

    # 1. SQL retrieval
    sql_desc, sql_df = router.route(question)
    parts.append(f"## SQL Query Result\n{sql_desc}")
    if not sql_df.empty:
        parts.append(sql_df.to_string(index=False))
    else:
        parts.append("(No structured data matched this query.)")

    # 2. Vector retrieval
    vector_context = vector.get_context(question, n_results=3)
    parts.append(f"\n## Relevant Documents (Semantic Search)\n{vector_context}")

    return "\n\n".join(parts)


def ask(question: str, verbose: bool = False) -> str:
    """Full hybrid RAG pipeline: route + retrieve + generate."""
    intent = classify_intent(question)
    context = build_context(question)

    if verbose:
        print(f"Intent: {intent}")
        print(f"District: {extract_district(question)}")
        print(f"Disease: {extract_disease(question)}")
        print(f"\n--- Context sent to LLM ---")
        print(context[:1000])
        print("..." if len(context) > 1000 else "")
        print("---\n")

    prompt = f"""Based on the following IDSP Kerala data, answer the user's question.

DATA CONTEXT:
{context}

USER QUESTION: {question}

ANSWER:"""

    response = gemini_client.models.generate_content(
        model=LLM_MODEL,
        contents=prompt,
        config={
            "system_instruction": SYSTEM_PROMPT,
            "temperature": 0.2,
            "max_output_tokens": 1024,
        }
    )

    return response.text


print("Hybrid RAG pipeline ready.")

Hybrid RAG pipeline ready.


## 4. Test the Chatbot

In [15]:
# Test with verbose output to see the full pipeline
answer = ask("What diseases are reported in Kannur?", verbose=True)
print("ANSWER:")
print(answer)

Intent: district_disease_summary
District: Kannur
Disease: None

--- Context sent to LLM ---
## SQL Query Result
Diseases reported in Kannur on 2026-09-01.

                 disease     metric subtype  value
Acute Diarrhoeal Disease  confirmed    None  169.0
              Chickenpox  confirmed    None    6.0
                  Dengue  confirmed    None    1.0
                  Dengue  suspected    None    5.0
                   Fever  inpatient    None    6.0
                   Fever outpatient    None  795.0
               Influenza  confirmed    None    2.0
           Leptospirosis  confirmed    None    1.0
           Leptospirosis  suspected    None    1.0


## Relevant Documents (Semantic Search)
[locality_report] Report date: 2026-09-01
District: Kannur
Disease: Dengue
Confirmed cases in district: 1 confirmed cases
Reported localities: CHERUKUNNU
Source: Kerala IDSP Daily Report, page 2 (locality section).

---

[district_disease_summary] Report date: 2026-09-01
District: Kannur (K

ClientError: 404 NOT_FOUND. {'error': {'code': 404, 'message': 'This model models/gemini-2.0-flash is no longer available. Please update your code to use models/gemini-3.6-flash for the latest features and improvements. We recommend you to use the Interactions API.', 'status': 'NOT_FOUND'}}

In [ ]:
test_questions = [
    "What is the latest report date?",
    "Which disease has the highest confirmed cases in Ernakulam?",
    "Where was dengue reported in Kerala?",
    "How many deaths were reported and from which diseases?",
    "Compare dengue across all districts",
    "What is the statewide situation of leptospirosis?",
    "Any H1N1 deaths reported?",
    "Which areas in Thiruvananthapuram have dengue cases?",
]

for q in test_questions:
    print(f"\n{'='*70}")
    print(f"Q: {q}")
    print(f"\nA: {ask(q)}")

## 5. Interactive Chat Loop

In [ ]:
print("IDSP Kerala Chatbot (type 'quit' to exit)")
print("="*50)

while True:
    question = input("\nYou: ").strip()
    if question.lower() in ("quit", "exit", "q"):
        print("Goodbye!")
        break
    if not question:
        continue
    answer = ask(question)
    print(f"\nAssistant: {answer}")

## 6. Export as Streamlit App

In [ ]:
app_code = '''"""IDSP Kerala Hybrid RAG Chatbot - Streamlit App"""

import os
import streamlit as st
from dotenv import load_dotenv
from google import genai
from query_router import QueryRouter, classify_intent, extract_district, extract_disease
from vector_search import VectorSearch

load_dotenv()

DB_PATH = os.path.join("notebooks", "data", "idsp_kerala.db")
CHROMA_PATH = os.path.join("data", "chroma_db")
LLM_MODEL = "gemini-3.6-flash"

SYSTEM_PROMPT = """You are an IDSP Kerala Disease Surveillance Assistant. You answer questions
about disease outbreaks, case counts, deaths, and localities in Kerala using
official IDSP (Integrated Disease Surveillance Programme) daily reports.

RULES:
1. Only use the data provided in the context below. Never invent numbers or facts.
2. Always mention the report date so the user knows how current the data is.
3. If the data says 0 cases, say so - do not skip diseases with zero values.
4. Reported cases are preliminary and may change after lab tests and death audits.
   Include this caveat when discussing deaths or confirmed cases.
5. If the context doesn't contain enough information to answer, say so clearly.
6. Keep answers concise but complete. Use bullet points for lists.
7. When mentioning localities, note they come from the IDSP locality section and
   represent where cases were reported, not necessarily the only affected areas.
"""


@st.cache_resource
def load_components():
    router = QueryRouter(DB_PATH)
    vector = VectorSearch(CHROMA_PATH)
    gemini_client = genai.Client(api_key=os.getenv("GEMINI_API_KEY"))
    return router, vector, gemini_client


def build_context(question, router, vector):
    parts = []
    sql_desc, sql_df = router.route(question)
    parts.append(f"## SQL Query Result\\n{sql_desc}")
    if not sql_df.empty:
        parts.append(sql_df.to_string(index=False))
    else:
        parts.append("(No structured data matched.)")
    vector_context = vector.get_context(question, n_results=3)
    parts.append(f"\\n## Relevant Documents\\n{vector_context}")
    return "\\n\\n".join(parts)


def get_answer(question, router, vector, gemini_client):
    context = build_context(question, router, vector)
    prompt = f"""Based on the following IDSP Kerala data, answer the user\\'s question.

DATA CONTEXT:
{context}

USER QUESTION: {question}

ANSWER:"""
    response = gemini_client.models.generate_content(
        model=LLM_MODEL,
        contents=prompt,
        config={
            "system_instruction": SYSTEM_PROMPT,
            "temperature": 0.2,
            "max_output_tokens": 1024,
        }
    )
    return response.text


# --- Streamlit UI ---
st.set_page_config(page_title="IDSP Kerala Chatbot", page_icon="🏥")
st.title("🏥 IDSP Kerala Disease Surveillance Chatbot")
st.caption("Ask questions about disease outbreaks, cases, deaths, and localities in Kerala.")

router, vector, gemini_client = load_components()

if "messages" not in st.session_state:
    st.session_state.messages = []

for msg in st.session_state.messages:
    with st.chat_message(msg["role"]):
        st.markdown(msg["content"])

if prompt := st.chat_input("Ask about Kerala IDSP data..."):
    st.session_state.messages.append({"role": "user", "content": prompt})
    with st.chat_message("user"):
        st.markdown(prompt)

    with st.chat_message("assistant"):
        with st.spinner("Analyzing IDSP data..."):
            answer = get_answer(prompt, router, vector, gemini_client)
        st.markdown(answer)

    st.session_state.messages.append({"role": "assistant", "content": answer})
'''

app_path = os.path.join(PROJECT_ROOT, "app.py")
with open(app_path, "w", encoding="utf-8") as f:
    f.write(app_code)
print(f"Saved Streamlit app to {app_path}")
print("\nRun with: streamlit run app.py")

In [ ]:
router.close()
print("Cleanup complete.")